In [4]:
import pandas as pd
from google.colab import drive
drive.mount('/drive')


train_df = pd.read_csv("/drive/MyDrive/train_banking77.csv")
test_df = pd.read_csv("/drive/MyDrive/test_Banking77.csv")



Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [ ]:
import torch
from torch.utils.data import Dataset

class IntentDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=64
    ):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoded = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx])
        }

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [5]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
train_dataset

Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 10003
})

In [6]:
train_dataset[0]

{'text': 'I am still waiting on my card?',
 'label': 11,
 'label_text': 'card_arrival'}

In [10]:
from datasets import ClassLabel

num_labels = train_df['label'].nunique()

train_dataset = train_dataset.cast_column(
    "label", ClassLabel(num_classes=num_labels)
)

dataset_split = train_dataset.train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="label"
)

Casting the dataset:   0%|          | 0/10003 [00:00<?, ? examples/s]

In [13]:
dataset_split["validation"] = dataset_split["test"]
del dataset_split["test"]
dataset_split

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 9002
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1001
    })
})

In [14]:
label_mapping = (
    train_df[["label", "label_text"]]
    .drop_duplicates()
    .sort_values("label")
)

label_mapping.head(10)

,label,label_text
9179,0,activate_my_card
1557,1,age_limit
9644,2,apple_pay_or_google_pay
4545,3,atm_support
1118,4,automatic_top_up
8613,5,balance_not_updated_after_bank_transfer
3301,6,balance_not_updated_after_cheque_or_cash_deposit
7005,7,beneficiary_not_allowed
2077,8,cancel_transfer
9515,9,card_about_to_expire
